# FoodLensVN — Gradio Demo (A1 / A2 / B1 / B2)

Launches `app/demo.py` on Kaggle. Pulls A1/A2 checkpoints + B2 adapter from HF, then exposes a public `gradio` share link (Kaggle blocks raw ports, so `share=True` is required).

**Setup before running:**
1. **Settings → Accelerator → GPU** (P100 or T4 ×2).
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** (read access).

In [ ]:
# Cell 1: Clone develop or pull latest
import os
%cd /kaggle/working/
REPO_URL = 'https://github.com/tamir39/vqa-viet-project.git'
REPO_DIR = 'vqa-viet-project'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch develop {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

In [ ]:
# Cell 2: Install deps via uv
!pip install -q uv
!uv sync --frozen 2>&1 | tail -10

In [ ]:
# Cell 3: HF login via Kaggle secret
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
print('HF login OK')

In [ ]:
# Cell 4: Path + env setup
import os, sys
ROOT = '/kaggle/working/vqa-viet-project'
os.environ['FOODLENS_DATA_DIR'] = f'{ROOT}/data/foodlensvn'
os.environ['MPLBACKEND'] = 'Agg'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
# Cell 5: Pull dataset (for sample test images you can drop into the demo)
!uv run python scripts/fetch_dataset.py --dest $FOODLENS_DATA_DIR
!uv run python scripts/build_dataset.py --data-dir $FOODLENS_DATA_DIR --output-dir data/processed --image-variant squared

In [ ]:
# Cell 6: Pull A1/A2 checkpoints + B2 adapter from HF (with hf_transfer for reliability)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!pip install -q hf_transfer

from huggingface_hub import snapshot_download
for name in ['A1', 'A2', 'B2']:
    snapshot_download(
        repo_id=f'Tamir39/foodlensvn-{name}',
        repo_type='model',
        local_dir=f'reports/{name}',
        token=os.environ['HF_TOKEN'],
        max_workers=4,
        etag_timeout=30,
    )
    print(f'{name}: pulled to reports/{name}/')

In [ ]:
# Cell 7: Launch the Gradio demo with --share so Kaggle gives you a public URL.
# First call to each track is slow (lazy load); subsequent calls reuse the cached model.
!uv run python app/demo.py --share